# TD04 — k plus proches voisins et réseau RBF

## Objectif

Deux modèles résolvent des données **non linéairement séparables** sans rétropropagation :

- **k-NN** (k plus proches voisins) : modèle *paresseux* — on ne s'entraîne pas, on répond en regardant les `k` points d'entraînement les plus proches de la requête (vote **dur**).
- **Réseau RBF** (fonctions à base radiale) : noyaux gaussiens centrés sur les points d'entraînement + couche de sortie linéaire. C'est un « k-NN adouci » : *tous* les voisins comptent, avec un poids qui décroît avec la distance (pondération **douce**).

Au programme :
1. Implémenter **k-NN** en numpy (distance + vote majoritaire).
2. Construire un **réseau RBF** en réutilisant la classe `MLP` du TD03 pour la couche de sortie.
3. **Comparer** leurs frontières de décision et le rôle de `k` et de `σ`.

> Le réseau RBF est un pont entre k-NN (partie 6.a du cours) et le perceptron (TD03) : sa sortie est un neurone linéaire entraîné sur les activations radiales $φ(x)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy

## 1. Classe `MLP` réutilisée (TD03)

La couche de sortie du RBF est un simple neurone **linéaire + sigmoïde** sur les activations $φ(x)$. C'est exactement ce que fait la classe `MLP` du TD03 avec une seule couche. On la **réutilise telle quelle** (copiée de `mlp.ipynb`) : il suffira de lui donner `Phi(X)` en entrée.

> Seule `fit_batch` (descente de gradient par lot) nous sert ici.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_prime(z):
    s = sigmoid(z)
    return s * (1.0 - s)


def tanh_prime(z):
    return 1.0 - np.tanh(z) ** 2


ACTIVATIONS = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "tanh": (np.tanh, tanh_prime),
    "linear": (lambda z: z, lambda z: np.ones_like(z)),
}

In [ ]:
class MLP:
    def __init__(self, sizes, activation="sigmoid", output_activation="linear", seed=0):
        rng = np.random.default_rng(seed)
        self.sizes = list(sizes)
        self.Ws = []
        self.bs = []
        for n_in, n_out in zip(sizes[:-1], sizes[1:]):
            self.Ws.append(rng.normal(0.0, 0.5, size=(n_out, n_in)))
            self.bs.append(np.zeros(n_out))
        self.activation = activation
        self.output_activation = output_activation

    def predict(self, X):
        return self.forward(X)[0][-1]

    def loss(self, X, y):
        return 0.5 * float(np.mean(np.sum((self.predict(X) - y) ** 2, axis=1)))

    def _step(self, X, y, lr):
        for l, (dW, db) in enumerate(self.backward(X, y)):
            self.Ws[l] -= lr * dW
            self.bs[l] -= lr * db

    def forward(self, X):
        A = X
        As = [A]
        Zs = []
        n = len(self.Ws)
        for l, (W, b) in enumerate(zip(self.Ws, self.bs)):
            Z = A @ W.T + b
            Zs.append(Z)
            name = self.output_activation if l == n - 1 else self.activation
            A = ACTIVATIONS[name][0](Z)
            As.append(A)
        return As, Zs

    def backward(self, X, y):
        As, Zs = self.forward(X)
        N = X.shape[0]
        n = len(self.Ws)
        delta = (As[-1] - y) * ACTIVATIONS[self.output_activation][1](Zs[-1])
        dWs, dbs = [], []
        for l in range(n - 1, -1, -1):
            dW = delta.T @ As[l] / N
            db = delta.sum(axis=0) / N
            dWs.insert(0, dW)
            dbs.insert(0, db)
            if l > 0:
                delta = (delta @ self.Ws[l]) * ACTIVATIONS[self.activation][1](Zs[l - 1])
        return list(zip(dWs, dbs))

    def fit_batch(self, X, y, X_test=None, y_test=None, lr=0.1, epochs=2000, patience=200):
        history = {"train": [], "test": []}
        best_test = np.inf
        best = None
        no_improve = 0
        for epoch in range(epochs):
            self._step(X, y, lr)
            history["train"].append(self.loss(X, y))
            if X_test is not None:
                te = self.loss(X_test, y_test)
                history["test"].append(te)
                if te < best_test:
                    best_test = te
                    best = (copy.deepcopy(self.Ws), copy.deepcopy(self.bs))
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        break
        if best is not None:
            self.Ws, self.bs = best
        return history

    def fit_online(self, X, y, X_test=None, y_test=None, lr=0.1, epochs=200, patience=50, seed=0):
        rng = np.random.default_rng(seed)
        history = {"train": [], "test": []}
        best_test = np.inf
        best = None
        no_improve = 0
        N = X.shape[0]
        for epoch in range(epochs):
            for i in rng.permutation(N):
                self._step(X[i:i + 1], y[i:i + 1], lr)
            history["train"].append(self.loss(X, y))
            if X_test is not None:
                te = self.loss(X_test, y_test)
                history["test"].append(te)
                if te < best_test:
                    best_test = te
                    best = (copy.deepcopy(self.Ws), copy.deepcopy(self.bs))
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        break
        if best is not None:
            self.Ws, self.bs = best
        return history

## 2. Jeu de données non linéaire

Deux classes concentriques dans le plan :
- **classe 0** (centre) : disque de rayon ≈ 0.5 ;
- **classe 1** (couronne) : anneau de rayon ≈ 1.0.

Aucune droite ne sépare ces deux classes (le perceptron linéaire du TD01 échoue). Un bruit gaussien radial est ajouté, et on sépare **train/test** pour mesurer la généralisation.

In [ ]:
def gen_circles(n=400, noise=0.1, seed=0):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n):
        if rng.random() < 0.5:
            r = 0.5 + rng.normal(0.0, noise)
            label = 0
        else:
            r = 1.0 + rng.normal(0.0, noise)
            label = 1
        theta = rng.uniform(0.0, 2.0 * np.pi)
        X.append([r * np.cos(theta), r * np.sin(theta)])
        y.append(label)
    return np.array(X), np.array(y).reshape(-1, 1)


def plot_points(X, y, ax=None, title=""):
    if ax is None:
        fig, ax = plt.subplots()
    ax.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], c="tab:blue", s=20,
               label="classe 0 (centre)", edgecolors="k", linewidths=0.5)
    ax.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], c="tab:red", s=20,
               label="classe 1 (couronne)", edgecolors="k", linewidths=0.5)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    return ax


X, y = gen_circles(n=400, noise=0.1, seed=0)

rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
n_train = int(0.7 * len(X))
X_train, y_train = X[idx[:n_train]], y[idx[:n_train]]
X_test, y_test = X[idx[n_train:]], y[idx[n_train:]]

print("X :", X.shape, "| train :", X_train.shape, "| test :", X_test.shape)
plot_points(X, y, title="Cercles concentriques (bruités)")

## 3. k plus proches voisins (k-NN)

**Principe** : pour classer une requête $q$, on cherche ses `k` plus proches voisins dans l'ensemble d'entraînement (distance euclidienne), puis on prend l'étiquette **majoritaire**.

Pas d'apprentissage : tout se fait à la prédiction (modèle *paresseux*).

> **Exercice 1** — Complétez `knn_predict` : calcul des distances puis vote majoritaire.

In [ ]:
def knn_predict(X, y, Xq, k=1):
    """Étiquette majoritaire des k plus proches voisins de chaque requête.

    X  : (N, d) points d'entraînement
    y  : (N, 1) étiquettes 0/1
    Xq : (M, d) requêtes
    """
    # BEGIN
    # distances[i, j] = || Xq[i] - X[j] ||  (diffusion (M, N, d))
    diff = Xq[:, None, :] - X[None, :, :]
    dists = np.sqrt((diff ** 2).sum(axis=2))
    # indices des k plus proches voisins pour chaque requête
    idx = np.argsort(dists, axis=1)[:, :k]
    votes = y[idx].reshape(idx.shape[0], k)
    # majorité : moyenne des étiquettes 0/1 (>= 0.5 -> classe 1)
    pred = (votes.mean(axis=1) >= 0.5).astype(float)
    # END
    return pred.reshape(-1, 1)

In [ ]:
def accuracy(y_true, y_pred):
    return float(np.mean(y_true.ravel() == y_pred.ravel()))


def plot_decision(predict_fn, X, y, ax=None, title=""):
    if ax is None:
        fig, ax = plt.subplots()
    x1 = np.linspace(X[:, 0].min() - 0.2, X[:, 0].max() + 0.2, 120)
    x2 = np.linspace(X[:, 1].min() - 0.2, X[:, 1].max() + 0.2, 120)
    xx, yy = np.meshgrid(x1, x2)
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = predict_fn(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors=["tab:blue", "tab:red"], alpha=0.3)
    ax.scatter(X[y.ravel() == 0, 0], X[y.ravel() == 0, 1], c="tab:blue", s=15, edgecolors="k")
    ax.scatter(X[y.ravel() == 1, 0], X[y.ravel() == 1, 1], c="tab:red", s=15, edgecolors="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.grid(False)
    return ax

Frontière de décision et précision pour quelques valeurs de `k`.

In [ ]:
ks = [1, 3, 10]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, k in zip(axes, ks):
    plot_decision(lambda g, k=k: knn_predict(X_train, y_train, g, k=k),
                  X_train, y_train, ax=ax, title=f"k-NN (k={k})")
    acc_tr = accuracy(y_train, knn_predict(X_train, y_train, X_train, k=k))
    acc_te = accuracy(y_test, knn_predict(X_train, y_train, X_test, k=k))
    print(f"k={k}: précision train={acc_tr:.3f}  test={acc_te:.3f}")
plt.tight_layout()

## 4. Réseau RBF (fonctions à base radiale)

**Principe** : on place un noyau gaussien sur **chaque point d'entraînement** (les centres $c_j$) :

$$φ_j(x) = \exp\!\left(-\frac{\|x - c_j\|^2}{2σ^2}\right), \qquad c_j = x_j^{(\text{train})}$$

La sortie est un neurone linéaire + sigmoïde sur ces activations :

$$s(x) = σ\!\left(w \cdot φ(x) + b\right), \qquad \hat y = [s(x) > 0.5]$$

Contrairement à k-NN qui ne garde que les `k` plus proches (vote dur), le RBF fait voter **tous** les centres, pondérés par $φ_j(x)$ qui décroît avec la distance (pondération douce). La largeur `σ` joue un rôle analogue à `k` : petit `σ` = voisinage étroit (sur-apprend), grand `σ` = voisinage large (lisse).

> **Exercice 2** — Complétez `rbf_kernel` (noyau gaussien) et `Phi` (matrice des activations).
> **Exercice 3** — Assemblez le RBF pour `σ = 0.5` : centres = points d'entraînement, entraînez `MLP([m, 1], sigmoid)` sur `Phi(X_train)`, prédisez au seuil 0.5.

In [ ]:
def rbf_kernel(x, c, sigma):
    """Noyau gaussien (fonction de base radiale) : phi_j(x) = exp(-||x - c||^2 / (2 sigma^2))."""
    # BEGIN
    return np.exp(-np.sum((x - c) ** 2) / (2.0 * sigma ** 2))
    # END


def Phi(X, centers, sigma):
    """Matrice des activations : Phi[i, j] = rbf_kernel(X[i], centers[j]).

    X       : (N, d) points à transformer
    centers : (m, d) centres (= points d'entraînement ici)
    """
    # BEGIN
    # diffusion (N, m, d) : toutes les différences X[i] - centers[j]
    diff = X[:, None, :] - centers[None, :, :]
    sq = (diff ** 2).sum(axis=2)          # (N, m) distances au carré
    return np.exp(-sq / (2.0 * sigma ** 2))
    # END

In [ ]:
# Assemblage du RBF pour sigma = 0.5 (Exercice 3)

# BEGIN
sigma = 0.5
centers = X_train
Phi_train = Phi(X_train, centers, sigma)
Phi_test = Phi(X_test, centers, sigma)

model = MLP([centers.shape[0], 1], output_activation="sigmoid", seed=0)
model.fit_batch(Phi_train, y_train, lr=0.5, epochs=2000)

pred_train = (model.predict(Phi_train) > 0.5).astype(float)
pred_test = (model.predict(Phi_test) > 0.5).astype(float)
# END

print("RBF σ=0.5 : précision train =", round(accuracy(y_train, pred_train), 3),
      " test =", round(accuracy(y_test, pred_test), 3))

In [ ]:
def train_rbf(X_train, y_train, sigma, lr=0.5, epochs=2000):
    """Construit Phi (centres = points d'entraînement) et entraîne la sortie linéaire."""
    centers = X_train
    Phi_train = Phi(X_train, centers, sigma)
    model = MLP([centers.shape[0], 1], output_activation="sigmoid", seed=0)
    model.fit_batch(Phi_train, y_train, lr=lr, epochs=epochs)
    return model, centers


def predict_rbf(model, centers, Xq, sigma):
    """Prédiction binaire du RBF (seuil 0.5 sur la sortie sigmoïde)."""
    Phi_q = Phi(Xq, centers, sigma)
    return (model.predict(Phi_q) > 0.5).astype(float)

## 5. Comparaison k-NN vs RBF

Frontières côte à côte : ligne du haut = k-NN (`k=1,3,10`), ligne du bas = RBF (`σ=0.1,0.5,1.0`).

> **Exercice 4** — Observez l'effet de `k` et de `σ` sur le **lissage** de la frontière. Quel réglage sur-apprend ? sous-apprend ? Que coûte une prédiction k-NN vs RBF ?

In [ ]:
ks = [1, 3, 10]
sigmas = [0.1, 0.5, 1.0]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

print("k-NN")
for ax, k in zip(axes[0], ks):
    plot_decision(lambda g, k=k: knn_predict(X_train, y_train, g, k=k),
                  X_train, y_train, ax=ax, title=f"k-NN (k={k})")
    acc_tr = accuracy(y_train, knn_predict(X_train, y_train, X_train, k=k))
    acc_te = accuracy(y_test, knn_predict(X_train, y_train, X_test, k=k))
    print(f"  k={k}: train={acc_tr:.3f}  test={acc_te:.3f}")

print("RBF")
for ax, sigma in zip(axes[1], sigmas):
    model, centers = train_rbf(X_train, y_train, sigma)
    plot_decision(lambda g, s=sigma: predict_rbf(model, centers, g, s),
                  X_train, y_train, ax=ax, title=f"RBF (σ={sigma})")
    acc_tr = accuracy(y_train, predict_rbf(model, centers, X_train, sigma))
    acc_te = accuracy(y_test, predict_rbf(model, centers, X_test, sigma))
    print(f"  σ={sigma}: train={acc_tr:.3f}  test={acc_te:.3f}")

plt.tight_layout()

## 6. À retenir

- **k-NN** est *paresseux* : tout le coût est à la prédiction (il faut balayer tout le jeu d'entraînement). Le RBF, une fois entraîné, ne stocke que ses centres et ses poids.
- **k-NN** = sélection **dure** des `k` voisins ; **RBF** = pondération **douce** de tous les voisins par un noyau gaussien. On peut voir le RBF comme un k-NN « lissé ».
- `k` et `σ` contrôlent le **lissage** : trop petit → frontière dentelée (sur-apprentissage) ; trop grand → frontière trop lisse (sous-apprentissage).
- Le RBF fait le pont entre k-NN et le perceptron : sa sortie est un neurone linéaire entraîné sur les activations radiales.

> **Bonus** : comparez avec le MLP du TD03 (couches cachées + rétropropagation) sur le même jeu de cercles. Qui généralise le mieux pour un nombre de paramètres donné ? Que devient le RBF si on choisit les centres par l'algorithme de Lloyd (partie 6.b du cours) ?